<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/KIMI2_TOPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TOPO

In [ ]:
# First, clean install of transformers 4.44.0
!pip uninstall transformers -y
!pip install transformers==4.48.2 accelerate bitsandbytes torch>=2.1.0 -q

In [1]:
!pip show transformers accelerate datasets torch bitsandbytes

Name: transformers
Version: 4.48.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers
---
Name: accelerate
Version: 1.14.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The Hugging Face team
Author-email: transformers@huggingface.co
License: Apache
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: peft
---
Name: datasets
Version: 4.0.0
Summary: HuggingFace community-d

In [5]:
# ============================================================================
# TOPOAI — MULTI-RUN SUITE (5 LR SWEEPS) — Kimi-VL-A3B-Thinking
# TOPO-2026 | Sovereign Machine Lab | Frank Morales Aguilera
# ============================================================================

import sys
import os
import gc
import copy
import time
import json
import shutil
import hashlib
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Dict, Tuple, Optional
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from google.colab import userdata
from datasets import load_dataset
from huggingface_hub import login, HfApi, create_repo, upload_folder

# Now import transformers AFTER installing the correct version
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*torch_dtype.*')

# ============================================================================
# MULTI-RUN CONFIGURATION
# ============================================================================
NUM_RUNS   = 5
FIXED_SEED = 123
PRIME_LIMIT = 13
EPOCHS     = 6
BATCH_SIZE = 8

LR_GRID = [
    (5e-3, 1e-3),
    (1e-3, 5e-4),
    (1e-2, 2e-3),
    (5e-3, 5e-3),
    (2e-3, 1e-3),
]
assert len(LR_GRID) == NUM_RUNS, "LR_GRID must have exactly NUM_RUNS entries."

YOUR_USERNAME  = 'frankmorales2020'
MODEL_NAME_HF  = 'topological-ai-Kimi-VL-A3B-Thinking-multirun'
REPO_ID        = f'{YOUR_USERNAME}/{MODEL_NAME_HF}'
BASE_MODEL_ID  = 'moonshotai/Kimi-VL-A3B-Thinking'
HIDDEN_SIZE    = 2048
SAMPLE_A, SAMPLE_B, SAMPLE_C = 500, 1000, 1000


# ----------------------------------------------------------------------------
# 1. CORE ARCHITECTURE WRAPPERS
# ----------------------------------------------------------------------------
class KimiLinear_TaskAwareModel(nn.Module):
    def __init__(self, base_model: nn.Module, hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]
        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

    def freeze_previous_heads(self, task: str):
        if task == 'B':
            self.classifier_A.requires_grad_(False)
        elif task == 'C':
            self.classifier_B.requires_grad_(False)

    def reset_heads(self):
        dev = next(self.base_model.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        for h in (self.classifier_A, self.classifier_B, self.classifier_C):
            h.requires_grad_(True)
        self.current_task = 'A'


# ----------------------------------------------------------------------------
# 2. TOPOLOGICAL GOVERNOR ENGINE
# ----------------------------------------------------------------------------
class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer = embed_layer
        vocab_size = embed_layer.weight.shape[0]
        sieve = [True] * (prime_limit + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(prime_limit ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, prime_limit + 1, i):
                    sieve[j] = False
        primes = [i for i in range(2, prime_limit + 1) if sieve[i]]
        self.anchor_indices = [p for p in primes if p < vocab_size]
        self.snapshot = {}
        self.safety_constant = 1.0 - np.prod([1.0 - (p ** -0.5) for p in self.anchor_indices])

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            return
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.snapshot:
            return True
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )

    def get_hash(self) -> str:
        hasher = hashlib.sha256()
        for idx in self.anchor_indices:
            weight_bytes = self.embed_layer.weight[idx].detach().float().cpu().numpy().tobytes()
            hasher.update(weight_bytes)
        return hasher.hexdigest()[:16]


# ----------------------------------------------------------------------------
# 3. DATASET UTILITIES
# ----------------------------------------------------------------------------
class AGNewsStreamDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


def prepare_tokenized_dataset(tokenizer, texts, labels) -> AGNewsStreamDataset:
    tokens = tokenizer(texts, max_length=64, padding='max_length',
                       truncation=True, return_tensors='pt')
    return AGNewsStreamDataset(tokens.input_ids, tokens.attention_mask,
                               torch.tensor(labels, dtype=torch.long))


# ----------------------------------------------------------------------------
# 4. TRAINING & EVALUATION
# ----------------------------------------------------------------------------
def train_task_explicit(
    task_label: str,
    model: KimiLinear_TaskAwareModel,
    dataset: AGNewsStreamDataset,
    embed_layer: nn.Embedding,
    governor: Optional[TopologicalGovernor] = None,
    epochs: int = EPOCHS,
    batch_size: int = BATCH_SIZE,
    lr_embed: float = 5e-3,
    lr_cls: float = 1e-3,
    run_id: int = 0
) -> float:
    model.switch_task(task_label)
    model.train()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    active_head = getattr(model, f'classifier_{task_label}')
    optimizer = torch.optim.AdamW([
        {'params': embed_layer.weight, 'lr': lr_embed},
        {'params': active_head.parameters(), 'lr': lr_cls}
    ])
    total_steps = epochs * len(dataloader)
    desc = f'[Run {run_id}] Task {task_label} | lr_embed={lr_embed:.0e} lr_cls={lr_cls:.0e}'
    progress_bar = tqdm(total=total_steps, desc=desc, leave=True)
    device = next(model.parameters()).device

    for epoch in range(epochs):
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            if governor:
                governor.zero_anchor_gradients()
            torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
            optimizer.step()
            if governor:
                governor.enforce_anchors()
            progress_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
            progress_bar.update(1)

    progress_bar.close()
    return evaluate_model_precision(model, dataloader)


def evaluate_model_precision(model: KimiLinear_TaskAwareModel,
                             dataloader: DataLoader) -> float:
    model.eval()
    correct = total = 0
    device = next(model.parameters()).device
    with torch.no_grad():
        for batch in dataloader:
            logits = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == batch['labels'].to(device)).sum().item()
            total += batch['labels'].size(0)
    return float(correct / total)


# ============================================================================
# 5. GLOBAL SETUP
# ============================================================================
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def full_vram_purge(objects_to_delete=None, sleep_secs=5):
    if objects_to_delete:
        for obj in objects_to_delete:
            if obj is not None:
                try:
                    if isinstance(obj, nn.Module):
                        obj.cpu()
                        for p in obj.parameters():
                            if p.grad is not None:
                                p.grad = None
                    del obj
                except:
                    pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    time.sleep(sleep_secs)


set_seed(FIXED_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('=' * 75)
print(f'TOPO-2026 MULTI-RUN INITIALISATION ({NUM_RUNS} runs, seed={FIXED_SEED})')
print('=' * 75)

# --- Dataset ---
print('\n[DATASET] Loading AG News splits...')
raw_ag_dataset = load_dataset('SetFit/ag_news', split='train')

def isolate_task_domain(dataset, class_labels, sample_limit):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled = filtered.select(range(min(sample_limit, len(filtered))))
    texts = [item['text'] for item in sampled]
    labels = [item['label'] % 2 for item in sampled]
    return texts, labels

task_a_texts, task_a_labels = isolate_task_domain(raw_ag_dataset, [0, 1], SAMPLE_A)
task_b_texts, task_b_labels = isolate_task_domain(raw_ag_dataset, [2, 3], SAMPLE_B)
task_c_texts, task_c_labels = isolate_task_domain(raw_ag_dataset, [0, 3], SAMPLE_C)

print('[DATASET] Loading AG News test split for held-out val evaluation...')
raw_ag_test = load_dataset('SetFit/ag_news', split='test')
val_a_texts, val_a_labels = isolate_task_domain(raw_ag_test, [0, 1], 200)
val_b_texts, val_b_labels = isolate_task_domain(raw_ag_test, [2, 3], 200)
val_c_texts, val_c_labels = isolate_task_domain(raw_ag_test, [0, 3], 200)

# --- Backbone ---
print('\n[BACKBONE] Materialising Kimi...')

# Clear any cached modules
if 'transformers_modules' in sys.modules:
    for mod in list(sys.modules.keys()):
        if 'transformers_modules' in mod:
            del sys.modules[mod]

# Load the model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).to(device)

# --- ROBUST MoE / AUX LOSS TRAINING PATCH ---
import types
for name, module in base_model.named_modules():
    if hasattr(module, "topk_method") or "Gate" in type(module).__name__ or "MoE" in type(module).__name__:
        orig_forward = module.forward
        def make_patched_forward(orig_f):
            def patched_forward(self, *args, **kwargs):
                prev_training = self.training
                self.training = False
                try:
                    res = orig_f(*args, **kwargs)
                    if isinstance(res, tuple) and len(res) == 3 and res[2] is None:
                        dev = args[0].device if len(args) > 0 else next(self.parameters()).device
                        res = (res[0], res[1], torch.tensor(0.0, device=dev, requires_grad=True))
                finally:
                    self.training = prev_training
                return res
            return patched_forward
        module.forward = types.MethodType(make_patched_forward(module.forward), module)

for param in base_model.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Locate embedding layer ---
if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
    embed_layer = base_model.transformer.wte
elif hasattr(base_model, 'model') and hasattr(base_model.model, 'embed_tokens'):
    embed_layer = base_model.model.embed_tokens
else:
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100000:
            embed_layer = module
            break
embed_layer.weight.requires_grad = True

# --- Pre-tokenise ---
dataset_A = prepare_tokenized_dataset(tokenizer, task_a_texts, task_a_labels)
dataset_B = prepare_tokenized_dataset(tokenizer, task_b_texts, task_b_labels)
dataset_C = prepare_tokenized_dataset(tokenizer, task_c_texts, task_c_labels)

val_dataset_A = prepare_tokenized_dataset(tokenizer, val_a_texts, val_a_labels)
val_dataset_B = prepare_tokenized_dataset(tokenizer, val_b_texts, val_b_labels)
val_dataset_C = prepare_tokenized_dataset(tokenizer, val_c_texts, val_c_labels)

# --- Model wrapper ---
model = KimiLinear_TaskAwareModel(base_model=base_model)
original_embed_weights = embed_layer.weight.detach().clone()

print('\n[READY] Setup complete. Starting multi-run sweep.')

# ============================================================================
# 6. MULTI-RUN SWEEP LOOP
# ============================================================================
run_results: List[Dict] = []
best_run_idx = -1
best_acc_c = -1.0
best_state_dict = None

for run_id in range(NUM_RUNS):
    lr_embed, lr_cls = LR_GRID[run_id]

    print('\n' + '=' * 75)
    print(f'  RUN {run_id + 1}/{NUM_RUNS}  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}')
    print('=' * 75)

    set_seed(FIXED_SEED)
    model.reset_heads()
    with torch.no_grad():
        embed_layer.weight.copy_(original_embed_weights)

    # TASK A
    print(f'\n[RUN {run_id}] TASK A: World vs Sports')
    train_task_explicit(
        'A', model, dataset_A, embed_layer,
        governor=None, lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id
    )
    _dl_train_a = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
    acc_a_initial = evaluate_model_precision(model, _dl_train_a)
    print(f'  [TASK A] Train Baseline: {acc_a_initial * 100:.2f}%')

    governor = TopologicalGovernor(embed_layer=embed_layer, prime_limit=PRIME_LIMIT)
    print(f'  [HIPPOCAMPUS] Anchoring {len(governor.anchor_indices)} prime coords: {governor.anchor_indices}')
    t0 = time.perf_counter()
    governor.take_snapshot()
    print(f'  [HIPPOCAMPUS] Snapshot in {(time.perf_counter()-t0)*1000:.2f} ms | hash={governor.get_hash()}')
    print(f'  [HIPPOCAMPUS] Safety Constant Λ (AST): {governor.safety_constant:.10f}')

    model.freeze_previous_heads('B')

    # TASK B
    print(f'\n[RUN {run_id}] TASK B: Business vs Sci/Tech')
    train_task_explicit(
        'B', model, dataset_B, embed_layer,
        governor=governor, lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id
    )
    _dl_train_b = DataLoader(dataset_B, batch_size=BATCH_SIZE, shuffle=False)
    acc_b_initial = evaluate_model_precision(model, _dl_train_b)
    print(f'  [TASK B] Train Baseline: {acc_b_initial * 100:.2f}%')

    model.freeze_previous_heads('C')

    # TASK C
    print(f'\n[RUN {run_id}] TASK C: World vs Sci/Tech')
    train_task_explicit(
        'C', model, dataset_C, embed_layer,
        governor=governor, lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id
    )
    _dl_val_c = DataLoader(val_dataset_C, batch_size=BATCH_SIZE, shuffle=False)
    acc_c_final = evaluate_model_precision(model, _dl_val_c)
    print(f'  [TASK C] Val Accuracy: {acc_c_final * 100:.2f}%')

    assert governor.verify_integrity(), f'[RUN {run_id}] Topological integrity violated!'

    # Forgetting measurement
    print(f'\n[RUN {run_id}] Measuring retention on training data...')
    dl_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
    dl_B = DataLoader(dataset_B, batch_size=BATCH_SIZE, shuffle=False)

    model.switch_task('A')
    acc_a_final = evaluate_model_precision(model, dl_A)
    print(f'  [TASK A] Final Train Accuracy: {acc_a_final * 100:.2f}%')

    model.switch_task('B')
    acc_b_final = evaluate_model_precision(model, dl_B)
    print(f'  [TASK B] Final Train Accuracy: {acc_b_final * 100:.2f}%')

    fgt_A = (acc_a_initial - acc_a_final) * 100
    fgt_B = (acc_b_initial - acc_b_final) * 100
    combined_fgt = (fgt_A + fgt_B) / 2.0
    anchor_kb = (len(governor.anchor_indices) * embed_layer.weight.shape[1] * 4) / 1024

    run_record = {
        'run_id': run_id,
        'lr_embed': lr_embed,
        'lr_cls': lr_cls,
        'acc_a_final': acc_a_final,
        'acc_b_final': acc_b_final,
        'acc_c_final': acc_c_final,
        'fgt_A': fgt_A,
        'fgt_B': fgt_B,
        'combined_fgt': combined_fgt,
        'anchor_kb': anchor_kb,
        'anchor_hash': governor.get_hash(),
    }
    run_results.append(run_record)

    print(f'\n  ┌──────────────────────────────────────────────────────────────────────────────┐')
    print(f'  │  RUN {run_id} SUMMARY  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}')
    print(f'  ├──────────────────────────────────────────────────────────────────────────────┤')
    print(f'  │  Task A  acc={acc_a_final*100:6.2f}%  fgt={fgt_A:+6.2f}%  (World vs Sports)')
    print(f'  │  Task B  acc={acc_b_final*100:6.2f}%  fgt={fgt_B:+6.2f}%  (Business vs Sci/Tech)')
    print(f'  │  Task C  acc={acc_c_final*100:6.2f}%            (World vs Sci/Tech)')
    print(f'  │  Combined Forgetting : {combined_fgt:+.2f}%')
    print(f'  │  Anchor Memory       : {anchor_kb:.2f} KB')
    print(f'  └──────────────────────────────────────────────────────────────────────────────┘')

    if acc_c_final > best_acc_c:
        best_acc_c = acc_c_final
        best_run_idx = run_id
        cpu_state = {k: v.cpu() for k, v in model.state_dict().items()}
        best_state_dict = copy.deepcopy(cpu_state)
        del cpu_state
        print(f'  ★ New best model saved (Run {run_id}, Task C: {acc_c_final*100:.2f}%)')

    # Memory purge
    print(f'\n[RUN {run_id}] Purging GPU memory...')
    if embed_layer.weight.grad is not None:
        embed_layer.weight.grad = None
    if governor is not None:
        governor.snapshot.clear()
    for _obj in [governor, dl_A, dl_B, _dl_train_a, _dl_train_b, _dl_val_c]:
        try:
            del _obj
        except Exception:
            pass
    full_vram_purge(objects_to_delete=None)
    if torch.cuda.is_available():
        alloc_gb = torch.cuda.memory_allocated() / 1024**3
        reserv_gb = torch.cuda.memory_reserved() / 1024**3
        print(f'  [PURGE] VRAM → allocated: {alloc_gb:.3f} GB | reserved: {reserv_gb:.3f} GB')

print('\n' + '=' * 75)
print('ALL RUNS COMPLETE')
print('=' * 75)

TOPO-2026 MULTI-RUN INITIALISATION (5 runs, seed=123)

[DATASET] Loading AG News splits...
[DATASET] Loading AG News test split for held-out val evaluation...

[BACKBONE] Materialising Kimi...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]


[READY] Setup complete. Starting multi-run sweep.

  RUN 1/5  |  lr_embed=5e-03  lr_cls=1e-03

[RUN 0] TASK A: World vs Sports


[Run 0] Task A | lr_embed=5e-03 lr_cls=1e-03:   0%|          | 0/378 [00:00<?, ?it/s]

  [TASK A] Train Baseline: 100.00%
  [HIPPOCAMPUS] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [HIPPOCAMPUS] Snapshot in 0.23 ms | hash=0b67f3f64ca457ef
  [HIPPOCAMPUS] Safety Constant Λ (AST): 0.9785142874

[RUN 0] TASK B: Business vs Sci/Tech


...

  [TASK A] Train Baseline: 100.00%
  [HIPPOCAMPUS] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [HIPPOCAMPUS] Snapshot in 0.21 ms | hash=9a044446172f5855
  [HIPPOCAMPUS] Safety Constant Λ (AST): 0.9785142874

[RUN 4] TASK B: Business vs Sci/Tech


[Run 4] Task B | lr_embed=2e-03 lr_cls=1e-03:   0%|          | 0/750 [00:00<?, ?it/s]

  [TASK B] Train Baseline: 99.50%

[RUN 4] TASK C: World vs Sci/Tech


[Run 4] Task C | lr_embed=2e-03 lr_cls=1e-03:   0%|          | 0/750 [00:00<?, ?it/s]

  [TASK C] Val Accuracy: 90.00%

[RUN 4] Measuring retention on training data...
  [TASK A] Final Train Accuracy: 98.80%
  [TASK B] Final Train Accuracy: 97.70%

  ┌──────────────────────────────────────────────────────────────────────────────┐
  │  RUN 4 SUMMARY  |  lr_embed=2e-03  lr_cls=1e-03
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  Task A  acc= 98.80%  fgt= +1.20%  (World vs Sports)
  │  Task B  acc= 97.70%  fgt= +1.80%  (Business vs Sci/Tech)
  │  Task C  acc= 90.00%            (World vs Sci/Tech)
  │  Combined Forgetting : +1.50%
  │  Anchor Memory       : 48.00 KB
  └──────────────────────────────────────────────────────────────────────────────┘
  ★ New best model saved (Run 4, Task C: 90.00%)

[RUN 4] Purging GPU memory...
  [PURGE] VRAM → allocated: 63.617 GB | reserved: 75.758 GB

ALL RUNS COMPLETE


## HF

In [7]:
# ============================================================================
# HUGGING FACE DEPLOYMENT CELL FOR KIMI TOPO-2026 CERTIFIED MODEL
# ============================================================================

import os
import json
import torch
from huggingface_hub import HfApi, create_repo

# Configuration
REPO_ID = f"{YOUR_USERNAME}/{MODEL_NAME_HF}"
LOCAL_DIR = "./topo_kimi_output"

# 1. Explicitly force creation of the local directory from scratch
if not os.path.exists(LOCAL_DIR):
    os.makedirs(LOCAL_DIR, exist_ok=True)
    print(f"[DIR] Successfully created missing directory: {LOCAL_DIR}")
else:
    print(f"[DIR] Directory already exists: {LOCAL_DIR}")

# 2. Save certification metadata and summary results from the multi-run sweep
certification_data = {
    "model": "Kimi-VL-A3B-Thinking",
    "framework": "TOPO-2026",
    "prime_anchors": [2, 3, 5, 7, 11, 13],
    "safety_constant": 0.9785142874,
    "anchor_memory_kb": 48.0,
    "best_task_c_accuracy": float(best_acc_c * 100),
    "best_run_id": best_run_idx,
    "all_runs_results": run_results,
    "status": "CERTIFIED"
}

cert_path = os.path.join(LOCAL_DIR, "certification.json")
with open(cert_path, "w") as f:
    json.dump(certification_data, f, indent=4)
print(f"[SAVE] Written metadata to {cert_path}")

# 3. Save the best model state dictionary from memory if available
if best_state_dict is not None:
    model_path = os.path.join(LOCAL_DIR, "classifier_heads.pt")
    torch.save(best_state_dict, model_path)
    print(f"[SAVE] Successfully saved best model state dictionary to {model_path}")

# 4. Confirm folder contents before pushing
print(f"[CHECK] Verified files in {LOCAL_DIR}:", os.listdir(LOCAL_DIR))

# 5. Create Hugging Face Repository and upload folder
print(f"[HF] Creating repository: {REPO_ID}...")
create_repo(REPO_ID, repo_type="model", exist_ok=True)

api = HfApi()
print(f"[HF] Uploading artifacts from {LOCAL_DIR} to {REPO_ID}...")
api.upload_folder(
    folder_path=LOCAL_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload TOPO-2026 multi-run certified Kimi-VL-A3B-Thinking artifacts and 5x5 metrics",
)

print(f"[HF] Deployment complete! View your model at: https://huggingface.co/{REPO_ID}")

[DIR] Directory already exists: ./topo_kimi_output
[SAVE] Written metadata to ./topo_kimi_output/certification.json
[SAVE] Successfully saved best model state dictionary to ./topo_kimi_output/classifier_heads.pt
[CHECK] Verified files in ./topo_kimi_output: ['certification.json', 'classifier_heads.pt']
[HF] Creating repository: frankmorales2020/topological-ai-Kimi-VL-A3B-Thinking-multirun...
[HF] Uploading artifacts from ./topo_kimi_output to frankmorales2020/topological-ai-Kimi-VL-A3B-Thinking-multirun...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...utput/classifier_heads.pt:   0%|          | 9.91MB / 32.8GB            

[HF] Deployment complete! View your model at: https://huggingface.co/frankmorales2020/topological-ai-Kimi-VL-A3B-Thinking-multirun


## INFERENCE

In [1]:
# ============================================================================
# INFERENCE CODE FOR TOPO-2026 CERTIFIED KIMI-VL-A3B-THINKING
# ============================================================================

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download

# Configuration
REPO_ID = "frankmorales2020/topological-ai-Kimi-VL-A3B-Thinking-multirun"
BASE_MODEL_ID = "moonshotai/Kimi-VL-A3B-Thinking"
HIDDEN_SIZE = 2048
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class KimiLinear_InferenceModel(nn.Module):
    def __init__(self, base_model: nn.Module, hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'C' # Default to terminal task C

    def forward(self, input_ids, attention_mask=None):
        with torch.no_grad():
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        hidden_states = outputs.hidden_states[-1]
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]
        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def set_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

# 1. Load Base Model and Tokenizer
print("[INFERENCE] Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).to(DEVICE)
base_model.eval()

# 2. Instantiate Task-Aware Wrapper
model = KimiLinear_InferenceModel(base_model=base_model).to(DEVICE)

# 3. Download and Load Certified Classifier Heads from Hugging Face Hub
print(f"[INFERENCE] Downloading certified heads from {REPO_ID}...")
weights_path = hf_hub_download(repo_id=REPO_ID, filename="classifier_heads.pt")
state_dict = torch.load(weights_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()
print("[INFERENCE] Certified model loaded successfully!")

# 4. Test Inference Function
def predict_text(text: str, task: str = 'C') -> str:
    model.set_task(task)
    tokens = tokenizer(text, max_length=64, padding='max_length',
                       truncation=True, return_tensors='pt')
    input_ids = tokens.input_ids.to(DEVICE)
    attention_mask = tokens.attention_mask.to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        pred_idx = torch.argmax(logits, dim=-1).item()

    # Task mapping labels
    labels_map = {
        'A': {0: "World", 1: "Sports"},
        'B': {0: "Business", 1: "Sci/Tech"},
        'C': {0: "World", 1: "Sci/Tech"}
    }
    return labels_map[task][pred_idx]

# Example test run
sample_text = "Scientists discover a new exoplanet orbiting a distant star system."
predicted_class = predict_text(sample_text, task='C')
print(f"\n[TEST] Sample Input: '{sample_text}'")
print(f"[TEST] Predicted Task C Class: {predicted_class}")

[INFERENCE] Loading base model and tokenizer...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

[INFERENCE] Downloading certified heads from frankmorales2020/topological-ai-Kimi-VL-A3B-Thinking-multirun...


classifier_heads.pt:   0%|          | 0.00/32.8G [00:00<?, ?B/s]

[INFERENCE] Certified model loaded successfully!

[TEST] Sample Input: 'Scientists discover a new exoplanet orbiting a distant star system.'
[TEST] Predicted Task C Class: Sci/Tech
